In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, LSTM, GRU
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# ---------------------------
# Hyperparameters
# ---------------------------
vocab_size = 10000
max_len = 200
embedding_dim = 128

# ---------------------------
# Load IMDb dataset
# ---------------------------
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# Pad sequences
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
x_train

array([[   5,   25,  100, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]], dtype=int32)

In [ ]:
# ---------------------------
# Model - Vanilla RNN
# ---------------------------
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# ---------------------------
# Model - Stacked RNN
# ---------------------------
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    SimpleRNN(64, return_sequences=True),
    SimpleRNN(32),
    Dense(1, activation="sigmoid")
])

In [ ]:
# ---------------------------
# Model - Vanilla LSTM
# ---------------------------
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

In [ ]:
# ---------------------------
# Model - Stacked LSTM
# ---------------------------
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1, activation="sigmoid")
])

In [ ]:
# ---------------------------
# Model - GRU
# ---------------------------
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    GRU(64),
    Dense(1, activation="sigmoid")
])

In [ ]:
# ---------------------------
# Compile
# ---------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# ---------------------------
# Train
# ---------------------------
model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.7046 - loss: 0.5368 - val_accuracy: 0.8494 - val_loss: 0.3702
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9012 - loss: 0.2499 - val_accuracy: 0.8730 - val_loss: 0.3111
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9417 - loss: 0.1587 - val_accuracy: 0.8744 - val_loss: 0.3527
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9659 - loss: 0.1040 - val_accuracy: 0.8754 - val_loss: 0.4163
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9743 - loss: 0.0759 - val_accuracy: 0.8508 - val_loss: 0.4290


In [ ]:
# ---------------------------
# Evaluate
# ---------------------------
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8444 - loss: 0.4511
Test Accuracy: 0.844760000705719


### Prediction

In [ ]:
from tensorflow.keras.datasets import imdb
import numpy as np

# Word index: word -> integer
word_index = imdb.get_word_index()

# Reverse mapping: integer -> word
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"

In [ ]:
# Pick any index you want
sample_id = 5

sample_review = x_test[sample_id]
true_label = y_test[sample_id]

In [ ]:
decoded_review = " ".join(
    reverse_word_index.get(i, "?") for i in sample_review if i != 0
)

print("Review Text:\n")
print(decoded_review[:1000])  # print first 1000 chars
print("\nTrue Label:", "Positive" if true_label == 1 else "Negative")

Review Text:

<START> i'm absolutely disgusted this movie isn't being sold all who love this movie should email disney and increase the demand for it they'd eventually have to sell it then i'd buy copies for everybody i know everything and everybody in this movie did a good job and i haven't figured out why disney hasn't put this movie on dvd or on vhs in rental stores at least i haven't seen any copies this is a wicked good movie and should be seen by all the kids in the new generation don't get to see it and i think they should it should at least be put back on the channel this movie doesn't deserve a cheap <UNK> it deserves the real thing i'm them now this movie will be on dvd

True Label: Positive


In [ ]:
# Model expects batch dimension
sample_input = np.expand_dims(sample_review, axis=0)

prediction = model.predict(sample_input)[0][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step


In [ ]:
sentiment = "Positive" if prediction >= 0.5 else "Negative"
confidence = prediction if prediction >= 0.5 else 1 - prediction

print("\nModel Prediction:")
print("Sentiment:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")


Model Prediction:
Sentiment: Positive
Confidence: 89.67 %
